# CartPole Discretization

This notebook builds a finite abstract MDP for `CartPole-v1`.

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import random
import torch
import pandas as pd
import sys
from pathlib import Path
import seaborn as sns
import matplotlib.pyplot as plt
from tqdm import tqdm
from matplotlib.colors import LogNorm
from sklearn.cluster import KMeans
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
from IPython.display import Image, display

# This starts from the current working directory
# and goes up until it finds the 'fogas_torch' folder or '.git'
def find_root(current_path, marker="fogas_torch"):
    current_path = Path(current_path).resolve()
    for parent in [current_path] + list(current_path.parents):
        if (parent / marker).exists():
            return parent
    return current_path

PROJECT_ROOT = find_root(Path.cwd())
print(f"Project root found at: {PROJECT_ROOT}")

# Add project root to sys.path so we can import local packages
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from fogas_torch import LinearMDP, PolicySolver, EnvDataCollector
from fogas_torch.algorithm import (
    FOGASSolverVectorized,
    FOGASOracleSolverVectorized,
    FOGASHyperOptimizer,
    FOGASEvaluator,
    FOGASDataset,
)
from fogas_torch.dataset_collection.dataset_analyzer import DatasetAnalyzer
from fogas_torch.fqi.fqi_solver import FQISolver
from fogas_torch.fqi.fqi_evaluator import FQIEvaluator

seed = 44
np.random.seed(seed) # Add this
random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Project root found at: /home/mauro/Desktop/EMAI/Ljubljana/Thesis/Code
Loading dataset from: /home/mauro/Desktop/EMAI/Ljubljana/Thesis/Code/datasets/20_to_100grid.csv
Using device: cpu


In [10]:
DATASET_PATH = PROJECT_ROOT / "datasets" / "cartpole_discrete.csv"
MODEL_DATASET_PATH = PROJECT_ROOT / "datasets" / "cartpole_abstract_exhaustive.csv"
CENTERS_PATH = PROJECT_ROOT / "datasets" / "cartpole_abstract_state_centers.csv"
MODEL_PATH = PROJECT_ROOT / "datasets" / "cartpole_abstract_mdp.pt"

print(f"Project root: {PROJECT_ROOT}")
print(f"Discrete dataset path: {DATASET_PATH}")
print(f"Model-identification dataset path: {MODEL_DATASET_PATH}")
print(f"Abstract centers path: {CENTERS_PATH}")
print(f"Abstract model path: {MODEL_PATH}")

Project root: /home/mauro/Desktop/EMAI/Ljubljana/Thesis/Code
Discrete dataset path: /home/mauro/Desktop/EMAI/Ljubljana/Thesis/Code/datasets/cartpole_discrete.csv
Model-identification dataset path: /home/mauro/Desktop/EMAI/Ljubljana/Thesis/Code/datasets/cartpole_abstract_exhaustive.csv
Abstract centers path: /home/mauro/Desktop/EMAI/Ljubljana/Thesis/Code/datasets/cartpole_abstract_state_centers.csv
Abstract model path: /home/mauro/Desktop/EMAI/Ljubljana/Thesis/Code/datasets/cartpole_abstract_mdp.pt


## Discretization design

CartPole observations are `(x, x_dot, theta, theta_dot)`.

Cartesian grid:
- `x`: 5 bins inside the failure range `[-2.4, 2.4]`
- `x_dot`: 5 bins inside a clipped range `[-3.0, 3.0]`
- `theta`: 7 bins inside the failure range `[-12 deg, 12 deg]`
- `theta_dot`: 7 bins inside a clipped range `[-3.5, 3.5]`

This gives `5 * 5 * 7 * 7 = 1225` non-terminal abstract states.

We also add two absorbing abstract states:
- `FAIL_STATE_ID`: episode terminated by violating CartPole constraints
- `SUCCESS_STATE_ID`: episode truncated by the Gymnasium time limit

Total abstract states: `1227`.

In [11]:
ENV_ID = "CartPole-v1"
GAMMA = 0.99
N_ACTIONS = 2

# Coarse but feasible grid for an exact abstract model.
STATE_BINS = np.array([5, 5, 7, 7], dtype=np.int64)
OBS_LIMITS = np.array([
    2.4,                  # cart position threshold
    3.0,                  # clipped cart velocity
    12.0 * np.pi / 180.0, # pole angle threshold in radians (of failure)
    3.5,                  # clipped pole angular velocity
], dtype=np.float64)

OBS_LOW = -OBS_LIMITS
OBS_HIGH = OBS_LIMITS
CORE_STATE_COUNT = int(np.prod(STATE_BINS))
FAIL_STATE_ID = CORE_STATE_COUNT
SUCCESS_STATE_ID = CORE_STATE_COUNT + 1
N_STATES = CORE_STATE_COUNT + 2

states = torch.arange(N_STATES, dtype=torch.int64)
actions = torch.arange(N_ACTIONS, dtype=torch.int64)

# e.g. in range [-2.4, 2.4] for cart position, we have 5 bins: [-2.4, -1.2), [-1.2, 0), [0, 1.2), [1.2, 2.4)
BIN_EDGES = [
    np.linspace(lo, hi, n_bins + 1, dtype=np.float64)
    for lo, hi, n_bins in zip(OBS_LOW, OBS_HIGH, STATE_BINS)
]
# mid point of each bin, e.g. for cart position: [-2.4, -1.2), [-1.2, 0), [0, 1.2), [1.2, 2.4) -> centers at [-1.8, -0.6, 0.6, 1.8]
BIN_CENTERS = [0.5 * (edges[:-1] + edges[1:]) for edges in BIN_EDGES]

# clips each component into the allowed range, e.g. if cart position is 3.0, it will be clipped to 2.4
def clip_obs(obs):
    obs = np.asarray(obs, dtype=np.float64)
    return np.clip(obs, OBS_LOW, OBS_HIGH)

# continuos obs -> multi-bin indices (i0, i1, i2, i3) 
# with formula width = (hi - lo) / n_bins, idx = floor((value - lo) / width), and then clipped to [0, n_bins-1]
# e.g. for cart position 0.5, it falls into the bin [0, 1.2) which is the 3rd bin (index 2)
def obs_to_multi_bin(obs):
    obs = clip_obs(obs)
    multi = []
    for value, lo, hi, n_bins in zip(obs, OBS_LOW, OBS_HIGH, STATE_BINS):
        width = (hi - lo) / n_bins
        idx = int(np.floor((value - lo) / width))
        idx = min(max(idx, 0), int(n_bins) - 1)
        multi.append(idx)
    return tuple(multi)

# converts multi-bin indices to a single state id using row-major order, 
# e.g. for (i0, i1, i2, i3) it computes i0 * (n_bins1 * n_bins2 * n_bins3) + i1 * (n_bins2 * n_bins3) + i2 * n_bins3 + i3
def multi_bin_to_state_id(multi_bin):
    return int(np.ravel_multi_index(multi_bin, STATE_BINS))

# inverse of multi_bin_to_state_id, converts a state id back to multi-bin indices
def state_id_to_multi_bin(state_id):
    return tuple(np.unravel_index(int(state_id), STATE_BINS))


def obs_to_state_id(obs):
    return multi_bin_to_state_id(obs_to_multi_bin(obs))

# converts a state id back to the center observation of the corresponding multi-bin
def state_id_to_center_obs(state_id):
    state_id = int(state_id)
    if state_id == FAIL_STATE_ID:
        return np.array([np.nan, np.nan, np.nan, np.nan], dtype=np.float64)
    if state_id == SUCCESS_STATE_ID:
        return np.array([np.nan, np.nan, np.nan, np.nan], dtype=np.float64)
    multi_bin = state_id_to_multi_bin(state_id)
    return np.array([BIN_CENTERS[d][idx] for d, idx in enumerate(multi_bin)], dtype=np.float64)


env = gym.make(ENV_ID)
initial_obs, _ = env.reset(seed=seed)
INITIAL_STATE_ID = obs_to_state_id(initial_obs)

print(f"Initial observation: {initial_obs}")
print(f"Initial abstract state id: {INITIAL_STATE_ID}")
print(f"Non-terminal abstract states: {CORE_STATE_COUNT}")
print(f"Total abstract states: {N_STATES}")
print(f"Action count: {N_ACTIONS}")

Initial observation: [-0.03774345 -0.02418869 -0.00942293  0.0469184 ]
Initial abstract state id: 612
Non-terminal abstract states: 1225
Total abstract states: 1227
Action count: 2


Building a table of representative continuous coordinates for every abstract state.

In [12]:
center_rows = []
for state_id in range(CORE_STATE_COUNT):
    center = state_id_to_center_obs(state_id)
    center_rows.append({
        "state_id": state_id,
        "x": center[0],
        "x_dot": center[1],
        "theta": center[2],
        "theta_dot": center[3],
    })

center_rows.append({"state_id": FAIL_STATE_ID, "x": np.nan, "x_dot": np.nan, "theta": np.nan, "theta_dot": np.nan})
center_rows.append({"state_id": SUCCESS_STATE_ID, "x": np.nan, "x_dot": np.nan, "theta": np.nan, "theta_dot": np.nan})

centers_df = pd.DataFrame(center_rows)
centers_df.head()

,state_id,x,x_dot,theta,theta_dot
0,0,-1.92,-2.4,-0.17952,-3.0
1,1,-1.92,-2.4,-0.17952,-2.0
2,2,-1.92,-2.4,-0.17952,-1.0
3,3,-1.92,-2.4,-0.17952,0.0
4,4,-1.92,-2.4,-0.17952,1.0


## Data Collection

### Mixed behavior policies for offline data

A purely random CartPole dataset tends to terminate quickly and gives poor coverage near balanced states.

To get a more useful abstract model, we mix:
- a simple angle-based heuristic,
- a random policy.

Policy selection is episode-based, matching the spirit of the existing mixed dataset notebooks.

In [14]:
def heuristic_policy(obs, rng):
    x, x_dot, theta, theta_dot = np.asarray(obs, dtype=np.float64)
    score = theta + 0.25 * theta_dot + 0.05 * x + 0.05 * x_dot
    return int(score > 0.0)


def random_policy(obs, rng):
    return int(rng.integers(0, N_ACTIONS))


POLICIES = {
    0: heuristic_policy,
    1: random_policy,
}
POLICY_NAMES = {
    0: "heuristic",
    1: "random",
}
POLICY_PROBS = np.array([0.7, 0.3], dtype=np.float64)


def collect_discretized_cartpole_dataset(
    total_steps=50000,
    max_episode_steps=500,
    seed=42,
    save_path=None,
):
    # making the environment inside the function
    env = gym.make(ENV_ID)
    rng = np.random.default_rng(seed)
    records = []
    policy_step_counts = {idx: 0 for idx in POLICIES}
    policy_episode_counts = {idx: 0 for idx in POLICIES}

    episode = 0
    collected = 0
    while collected < total_steps:
        # still sensible to set the seed for each episode to have reproducibility
        obs, _ = env.reset(seed=seed + episode)
        # choose policy
        policy_id = int(rng.choice(list(POLICIES.keys()), p=POLICY_PROBS))
        policy_episode_counts[policy_id] += 1

        step = 0
        done = False
        while not done and collected < total_steps and step < max_episode_steps:
            # convert obs to abstract state id
            state_id = obs_to_state_id(obs)
            action = POLICIES[policy_id](obs, rng)
            # next step in the environment
            next_obs, reward, terminated, truncated, _ = env.step(action)

            # the pole fell or the cart left the allowed range
            if terminated:
                next_state_id = FAIL_STATE_ID
            # the episode ended due to time limit but not failure
            elif truncated:
                next_state_id = SUCCESS_STATE_ID
            else:
                next_state_id = obs_to_state_id(next_obs)

            records.append({
                "episode": episode,
                "step": step,
                "state": state_id,
                "action": action,
                "reward": float(reward),
                "next_state": next_state_id,
                "policy_id": policy_id,
                "terminated": bool(terminated),
                "truncated": bool(truncated),
            })

            policy_step_counts[policy_id] += 1
            collected += 1
            step += 1
            done = bool(terminated or truncated)
            obs = next_obs

        episode += 1

    env.close()

    df = pd.DataFrame(records)
    if save_path is not None:
        save_path.parent.mkdir(parents=True, exist_ok=True)
        df[["state", "action", "reward", "next_state"]].to_csv(save_path, index=False)

    print(f"Collected transitions: {len(df)}")
    print(f"Episodes: {df['episode'].nunique()}")
    print(f"Visited abstract states: {df['state'].nunique()} / {CORE_STATE_COUNT}")
    print(f"Terminal failures: {int(df['terminated'].sum())}")
    print(f"Time-limit truncations: {int(df['truncated'].sum())}")
    print("Policy usage by steps:")
    for idx, count in policy_step_counts.items():
        print(f"  {idx} ({POLICY_NAMES[idx]}): {count}")
    print("Policy usage by episodes:")
    for idx, count in policy_episode_counts.items():
        print(f"  {idx} ({POLICY_NAMES[idx]}): {count}")

    return df

df_cartpole = collect_discretized_cartpole_dataset(
    total_steps=50000,
    max_episode_steps=500,
    seed=seed,
    save_path=DATASET_PATH,
)

df_cartpole.head()


Collected transitions: 50000
Episodes: 134
Visited abstract states: 54 / 1225
Terminal failures: 35
Time-limit truncations: 98
Policy usage by steps:
  0 (heuristic): 49127
  1 (random): 873
Policy usage by episodes:
  0 (heuristic): 99
  1 (random): 35


,episode,step,state,action,reward,next_state,policy_id,terminated,truncated
0,0,0,612,0,1.0,612,0,False,False
1,0,1,612,1,1.0,612,0,False,False
2,0,2,612,1,1.0,612,0,False,False
3,0,3,612,0,1.0,612,0,False,False
4,0,4,612,1,1.0,612,0,False,False


### Data to infer P matrix and r

For estimating the abstract model, we use a deliberately optimistic synthetic dataset: one transition for every `(abstract state, action)` pair, starting from the center of each bin. This is not a realistic behavioral dataset, but it removes coverage issues and gives a much cleaner estimate of `P` and `r` for the discretized MDP itself.

In [15]:
def collect_exhaustive_transitions(save_path=None):
    env = gym.make(ENV_ID)
    records = []

    for state_id in range(CORE_STATE_COUNT):
        initial_obs = state_id_to_center_obs(state_id)

        for action in range(N_ACTIONS):
            env.reset()
            env.unwrapped.state = initial_obs.copy()

            next_obs, reward, terminated, truncated, _ = env.step(action)

            if terminated:
                next_state_id = FAIL_STATE_ID
            elif truncated:
                next_state_id = SUCCESS_STATE_ID
            else:
                next_state_id = obs_to_state_id(next_obs)

            records.append({
                "state": state_id,
                "action": action,
                "reward": float(reward),
                "next_state": next_state_id,
                "synthetic": True,
            })

    env.close()

    df = pd.DataFrame(records)
    if save_path is not None:
        save_path.parent.mkdir(parents=True, exist_ok=True)
        df[["state", "action", "reward", "next_state"]].to_csv(save_path, index=False)

    print(f"Collected exhaustive abstract transitions: {len(df)}")
    print(f"Covered core state-action pairs: {df[['state', 'action']].drop_duplicates().shape[0]} / {CORE_STATE_COUNT * N_ACTIONS}")
    print(f"Failure transitions: {(df['next_state'] == FAIL_STATE_ID).sum()}")
    print(f"Success transitions: {(df['next_state'] == SUCCESS_STATE_ID).sum()}")

    return df


df_cartpole_model = collect_exhaustive_transitions(save_path=MODEL_DATASET_PATH)

df_cartpole_model.head()

Collected exhaustive abstract transitions: 2450
Covered core state-action pairs: 2450 / 2450
Failure transitions: 200
Success transitions: 0


,state,action,reward,next_state,synthetic
0,0,0,1.0,1225,True
1,0,1,1.0,1225,True
2,1,0,1.0,1225,True
3,1,1,1.0,1225,True
4,2,0,1.0,2,True


## Empirical abstract MDP

We estimate:
- `P_hat[(s, a), s_next]` from transition counts,
- `r_hat[(s, a)]` from average reward.

The estimation dataset is `df_cartpole_model`, the exhaustive synthetic transition table built from bin centers.

For unseen abstract state-action pairs, we still keep a conservative fallback:
- transition to `FAIL_STATE_ID`,
- reward `0.0`.

Both terminal abstract states are made absorbing with reward `0.0`.


This function builds the actual Transition Matrix ($\hat{P}$) and Reward Vector ($\hat{r}$) based on the data (df).
- Counting: it loops through the data and counts how many times each (State, Action) pair led to a specific (Next State).
- Normalization: It divides those counts by the total number of times you visited that state.
- Handling the Unknown: If a state-action pair was never visited (unseen_pairs), it assumes the worst: it marks the transition as leading straight to the FAIL_STATE.
- Absorbing States: It ensures that once if is in a "Fail" or "Success" state, stay there (probability = 1.0).

In [17]:
def build_empirical_abstract_mdp(df):
    row_count = N_STATES * N_ACTIONS
    transition_counts = torch.zeros((row_count, N_STATES), dtype=torch.float64)
    reward_sums = torch.zeros(row_count, dtype=torch.float64)
    sa_counts = torch.zeros(row_count, dtype=torch.float64)

    for row in df.itertuples(index=False):
        idx = int(row.state) * N_ACTIONS + int(row.action)
        transition_counts[idx, int(row.next_state)] += 1.0
        reward_sums[idx] += float(row.reward)
        sa_counts[idx] += 1.0

    P_hat = torch.zeros_like(transition_counts)
    r_hat = torch.zeros(row_count, dtype=torch.float64)
    unseen_pairs = []

    for state_id in range(N_STATES):
        for action_id in range(N_ACTIONS):
            idx = state_id * N_ACTIONS + action_id

            if state_id == FAIL_STATE_ID:
                P_hat[idx, FAIL_STATE_ID] = 1.0
                r_hat[idx] = 0.0
                continue

            if state_id == SUCCESS_STATE_ID:
                P_hat[idx, SUCCESS_STATE_ID] = 1.0
                r_hat[idx] = 0.0
                continue

            count = sa_counts[idx].item()
            if count > 0:
                P_hat[idx] = transition_counts[idx] / count
                r_hat[idx] = reward_sums[idx] / count
            else:
                unseen_pairs.append((state_id, action_id))
                P_hat[idx, FAIL_STATE_ID] = 1.0
                r_hat[idx] = 0.0

    return P_hat, r_hat, sa_counts, unseen_pairs

In [18]:
def check_cartpole_abstract_mdp_logic(P_hat, r_hat, sa_counts, min_count=5):
    one_vec = torch.ones(P_hat.shape[0], dtype=torch.float64)
    row_sums_ok = bool(torch.allclose(P_hat.sum(dim=1), one_vec))
    nonnegative_ok = bool(torch.all(P_hat >= 0.0))

    fail_absorbing_ok = True
    success_absorbing_ok = True
    for action_id in range(N_ACTIONS):
        fail_idx = FAIL_STATE_ID * N_ACTIONS + action_id
        success_idx = SUCCESS_STATE_ID * N_ACTIONS + action_id
        fail_absorbing_ok &= bool(P_hat[fail_idx, FAIL_STATE_ID].item() == 1.0 and torch.count_nonzero(P_hat[fail_idx]).item() == 1)
        success_absorbing_ok &= bool(P_hat[success_idx, SUCCESS_STATE_ID].item() == 1.0 and torch.count_nonzero(P_hat[success_idx]).item() == 1)

    observed_mask = sa_counts[:CORE_STATE_COUNT * N_ACTIONS] > 0
    observed_rewards = r_hat[:CORE_STATE_COUNT * N_ACTIONS][observed_mask]
    mean_reward = float(observed_rewards.mean().item()) if observed_rewards.numel() > 0 else float('nan')
    min_reward = float(observed_rewards.min().item()) if observed_rewards.numel() > 0 else float('nan')
    max_reward = float(observed_rewards.max().item()) if observed_rewards.numel() > 0 else float('nan')

    low_count_pairs = int(((sa_counts[:CORE_STATE_COUNT * N_ACTIONS] > 0) & (sa_counts[:CORE_STATE_COUNT * N_ACTIONS] < min_count)).sum().item())

    action_delta_x = {0: [], 1: []}
    action_delta_theta = {0: [], 1: []}
    edge_fail_probs = []
    center_fail_probs = []
    success_probs = []

    finite_centers = np.stack([state_id_to_center_obs(s) for s in range(CORE_STATE_COUNT)], axis=0)

    for state_id in range(CORE_STATE_COUNT):
        center_obs = finite_centers[state_id]
        multi_bin = state_id_to_multi_bin(state_id)
        is_edge = any(idx == 0 or idx == int(STATE_BINS[d]) - 1 for d, idx in enumerate(multi_bin))
        is_center = all(0 < idx < int(STATE_BINS[d]) - 1 for d, idx in enumerate(multi_bin))

        for action_id in range(N_ACTIONS):
            idx = state_id * N_ACTIONS + action_id
            count = sa_counts[idx].item()
            if count <= 0:
                continue

            fail_prob = float(P_hat[idx, FAIL_STATE_ID].item())
            success_prob = float(P_hat[idx, SUCCESS_STATE_ID].item())
            success_probs.append(success_prob)
            if is_edge:
                edge_fail_probs.append(fail_prob)
            if is_center:
                center_fail_probs.append(fail_prob)

            nonterminal_probs = P_hat[idx, :CORE_STATE_COUNT].cpu().numpy()
            total_nonterminal = nonterminal_probs.sum()
            if total_nonterminal > 0:
                expected_next = (nonterminal_probs[:, None] * finite_centers).sum(axis=0) / total_nonterminal
                action_delta_x[action_id].append(float(expected_next[0] - center_obs[0]))
                action_delta_theta[action_id].append(float(expected_next[2] - center_obs[2]))

    mean_dx_left = float(np.mean(action_delta_x[0])) if action_delta_x[0] else float('nan')
    mean_dx_right = float(np.mean(action_delta_x[1])) if action_delta_x[1] else float('nan')
    mean_dtheta_left = float(np.mean(action_delta_theta[0])) if action_delta_theta[0] else float('nan')
    mean_dtheta_right = float(np.mean(action_delta_theta[1])) if action_delta_theta[1] else float('nan')
    edge_fail_mean = float(np.mean(edge_fail_probs)) if edge_fail_probs else float('nan')
    center_fail_mean = float(np.mean(center_fail_probs)) if center_fail_probs else float('nan')
    max_success_prob = float(np.max(success_probs)) if success_probs else float('nan')

    print("Logical sanity checks for the abstract CartPole model")
    print("-" * 60)
    print(f"Row sums valid: {row_sums_ok}")
    print(f"No negative probabilities: {nonnegative_ok}")
    print(f"Fail state absorbing: {fail_absorbing_ok}")
    print(f"Success state absorbing: {success_absorbing_ok}")
    print(f"Observed-pair reward mean/min/max: {mean_reward:.4f} / {min_reward:.4f} / {max_reward:.4f}")
    print(f"Observed pairs with count < {min_count}: {low_count_pairs}")
    print(f"Mean Δx for action 0 (left):  {mean_dx_left:.4f}")
    print(f"Mean Δx for action 1 (right): {mean_dx_right:.4f}")
    print(f"Mean Δtheta for action 0: {mean_dtheta_left:.4f}")
    print(f"Mean Δtheta for action 1: {mean_dtheta_right:.4f}")
    print(f"Mean fail prob on edge bins:   {edge_fail_mean:.4f}")
    print(f"Mean fail prob on center bins: {center_fail_mean:.4f}")
    print(f"Max success prob from any abstract pair: {max_success_prob:.4f}")

    print("\nInterpretation notes:")
    print("  * Rewards should be close to 1.0 on observed non-terminal pairs, since CartPole gives +1 per step.")
    print("  * Action 0 and action 1 should induce opposite average motion in x; otherwise the abstraction is probably too coarse.")
    print("  * Edge bins should have higher failure probability than central bins.")
    print("  * Success probability is not fully Markov here because CartPole truncation depends on elapsed time, which is not in the state.")

    return {
        "row_sums_ok": row_sums_ok,
        "nonnegative_ok": nonnegative_ok,
        "fail_absorbing_ok": fail_absorbing_ok,
        "success_absorbing_ok": success_absorbing_ok,
        "mean_reward": mean_reward,
        "min_reward": min_reward,
        "max_reward": max_reward,
        "low_count_pairs": low_count_pairs,
        "mean_dx_left": mean_dx_left,
        "mean_dx_right": mean_dx_right,
        "mean_dtheta_left": mean_dtheta_left,
        "mean_dtheta_right": mean_dtheta_right,
        "edge_fail_mean": edge_fail_mean,
        "center_fail_mean": center_fail_mean,
        "max_success_prob": max_success_prob,
    }

In [19]:
P_hat, r_hat, sa_counts, unseen_pairs = build_empirical_abstract_mdp(df_cartpole_model)

observed_pairs = int((sa_counts > 0).sum().item())
core_pairs = CORE_STATE_COUNT * N_ACTIONS
coverage_ratio = observed_pairs / core_pairs

print(f"Observed core state-action pairs: {observed_pairs} / {core_pairs} ({coverage_ratio:.2%})")
print(f"Unseen core state-action pairs: {len(unseen_pairs)}")
print(f"Transition matrix shape: {tuple(P_hat.shape)}")
print(f"Reward vector shape: {tuple(r_hat.shape)}")
print(f"Terminal abstract states: fail={FAIL_STATE_ID}, success={SUCCESS_STATE_ID}")
print(f"Row sums valid: {bool(torch.allclose(P_hat.sum(dim=1), torch.ones(P_hat.shape[0], dtype=torch.float64)))}")
print(f"Any negative probabilities: {bool(torch.any(P_hat < 0.0))}")

logic_metrics = check_cartpole_abstract_mdp_logic(P_hat, r_hat, sa_counts, min_count=5)

Observed core state-action pairs: 2450 / 2450 (100.00%)
Unseen core state-action pairs: 0
Transition matrix shape: (2454, 1227)
Reward vector shape: (2454,)
Terminal abstract states: fail=1225, success=1226
Row sums valid: True
Any negative probabilities: False
Logical sanity checks for the abstract CartPole model
------------------------------------------------------------
Row sums valid: True
No negative probabilities: True
Fail state absorbing: True
Success state absorbing: True
Observed-pair reward mean/min/max: 1.0000 / 1.0000 / 1.0000
Observed pairs with count < 5: 2450
Mean Δx for action 0 (left):  0.0000
Mean Δx for action 1 (right): 0.0000
Mean Δtheta for action 0: -0.0000
Mean Δtheta for action 1: -0.0000
Mean fail prob on edge bins:   0.1000
Mean fail prob on center bins: 0.0000
Max success prob from any abstract pair: 0.0000

Interpretation notes:
  * Rewards should be close to 1.0 on observed non-terminal pairs, since CartPole gives +1 per step.
  * Action 0 and action 1 s

In [ ]:
torch.save(
    {
        "env_id": ENV_ID,
        "gamma": GAMMA,
        "state_bins": STATE_BINS,
        "obs_low": OBS_LOW,
        "obs_high": OBS_HIGH,
        "core_state_count": CORE_STATE_COUNT,
        "fail_state_id": FAIL_STATE_ID,
        "success_state_id": SUCCESS_STATE_ID,
        "P": P_hat,
        "r": r_hat,
        "sa_counts": sa_counts,
        "initial_state_id": INITIAL_STATE_ID,
    },
    MODEL_PATH,
)

CENTERS_PATH.parent.mkdir(parents=True, exist_ok=True)
centers_df.to_csv(CENTERS_PATH, index=False)

print(f"Saved discrete dataset to: {DATASET_PATH}")
print(f"Saved model-identification dataset to: {MODEL_DATASET_PATH}")
print(f"Saved abstract centers to: {CENTERS_PATH}")
print(f"Saved abstract MDP tensors to: {MODEL_PATH}")

## Discrete-state compatibility object

This is only a compatibility wrapper around the abstract model.

For now we keep tabular one-hot features. Later, you can replace `phi_tabular` with an RBF feature map built on the exported abstract-state centers.

In [ ]:
def phi_tabular(x, a):
    vec = torch.zeros(N_STATES * N_ACTIONS, dtype=torch.float64)
    vec[int(x) * N_ACTIONS + int(a)] = 1.0
    return vec


abstract_mdp = LinearMDP(
    states=states,
    actions=actions,
    phi=phi_tabular,
    omega=r_hat.clone(),
    gamma=GAMMA,
    x0=INITIAL_STATE_ID,
    P=P_hat,
    terminal_states=[FAIL_STATE_ID, SUCCESS_STATE_ID],
)

print(f"Abstract MDP states: {abstract_mdp.N}")
print(f"Actions: {abstract_mdp.A}")
print(f"Feature dimension: {abstract_mdp.d}")
print(f"Feature norm bound R: {abstract_mdp.R.item():.3f}")

## Next step for RBF features

The exported file `cartpole_abstract_state_centers.csv` gives one representative continuous observation per abstract state.

A natural next notebook step is:
- define `phi_state(state_id)` from these center observations,
- place RBF centers on the abstract-state centers or on a subset of them,
- keep the same action coupling used in your grid notebooks: `phi(state, action) = one_hot(action) kron phi_state(state)`.

At that point, the discrete CSV dataset created here will already be usable by the current FOGAS data loader.